# Install test — P3M binaries (instructor only)

Not for students. This measures how long a plain `install.packages()` takes on
Colab when pointed at Posit Public Package Manager, which serves **pre-compiled**
Linux binaries instead of source.

Runtime observed 2026-09: R 4.6.1, Ubuntu 22.04.5 LTS (jammy).

**Run this on a fresh runtime** (*Runtime → Disconnect and delete runtime* first),
otherwise you are timing a machine that already has things cached.

Report back: the two elapsed times and whether every version printed.

## 1. Confirm the runtime still matches

The repository URL below hard-codes `jammy`. If this cell reports a different
Ubuntu release, Google has reimaged Colab and the URL must change.

In [ ]:
cat(R.version.string, "\n")
cat(system("grep PRETTY_NAME /etc/os-release | cut -d'\"' -f2", intern = TRUE), "\n")
cat("codename:",
    system("grep VERSION_CODENAME /etc/os-release | cut -d= -f2", intern = TRUE), "\n")

## 2. Point R at the binary repository

Two things matter here:

- the `__linux__/jammy` path, which is what makes P3M serve binaries rather than
  source tarballs
- the `HTTPUserAgent` option — P3M decides what to serve partly from the user
  agent string, and R's default does not always carry the platform

In [ ]:
options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"),
  HTTPUserAgent = sprintf(
    "R/%s R (%s)",
    getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])
  ),
  Ncpus = max(1L, parallel::detectCores())
)

cat("repo: ", getOption("repos")[["CRAN"]], "\n")
cat("cores:", getOption("Ncpus"), "\n")

## 3. Are we really being offered binaries?

Cheap check before committing to a long install. A binary repo lists a
`File` column pointing at a `.tar.gz` built for this platform; a source-only
repo does not.

In [ ]:
ap <- available.packages()
cat("packages visible:", nrow(ap), "\n")
cat("columns:", paste(colnames(ap), collapse = ", "), "\n\n")

if ("Seurat" %in% rownames(ap)) {
  cat("Seurat offered at version", ap["Seurat", "Version"], "\n")
  if ("File" %in% colnames(ap)) cat("File:", ap["Seurat", "File"], "\n")
} else {
  cat("Seurat NOT found in this repository — stop and report this.\n")
}

## 4. Timed install — the core CRAN packages

This is the number that decides whether we need a pre-built tarball at all.

In [ ]:
pkgs <- c("Seurat", "harmony", "patchwork", "dplyr", "ggplot2", "hdf5r")

elapsed <- system.time(
  install.packages(pkgs)
)[["elapsed"]]

cat("\n=== core install took", round(elapsed / 60, 1), "minutes ===\n")

## 5. Timed install — Bioconductor, and Canek

`glmGamPoi` speeds up `SCTransform`. It is on Bioconductor, not CRAN, so it
comes through `BiocManager`.

`Canek` **is** on CRAN (0.3.1, July 2026), but it imports `bluster`, which is
Bioconductor. A plain `install.packages("Canek")` against a CRAN-only repo would
therefore fail on that dependency. Going through `BiocManager` keeps both
repositories in play while still taking the CRAN binaries from P3M.


In [ ]:
elapsed_bioc <- system.time({
  install.packages("BiocManager")
  BiocManager::install(c("glmGamPoi", "Canek"), update = FALSE, ask = FALSE)
})[["elapsed"]]

cat("\n=== glmGamPoi + Canek took", round(elapsed_bioc / 60, 1), "minutes ===\n")


## 6. Does it all load?

Installing is not the same as loading. A missing *system* library (outside R)
shows up here and nowhere earlier.

In [ ]:
check <- c(pkgs, "glmGamPoi", "Canek")

for (p in check) {
  ok <- suppressWarnings(suppressMessages(
    require(p, character.only = TRUE, quietly = TRUE)
  ))
  cat(sprintf("%-12s %s  %s\n",
              p,
              if (ok) "OK  " else "FAIL",
              if (ok) as.character(packageVersion(p)) else ""))
}

## 7. Does Seurat actually run?

A tiny synthetic object through the standard pipeline. Catches a broken BLAS or
a missing C++ dependency that only bites at runtime.

In [ ]:
set.seed(1)
counts <- matrix(rpois(200 * 100, lambda = 2), nrow = 200)
rownames(counts) <- paste0("gene", seq_len(200))
colnames(counts) <- paste0("cell", seq_len(100))

obj <- CreateSeuratObject(counts = counts)
obj <- NormalizeData(obj, verbose = FALSE)
obj <- FindVariableFeatures(obj, verbose = FALSE)
obj <- ScaleData(obj, verbose = FALSE)
obj <- RunPCA(obj, npcs = 10, verbose = FALSE)

print(obj)
cat("\nSeurat pipeline ran end to end.\n")

## 7b. Does Canek run?

Two small synthetic batches with an offset between them, corrected with
`RunCanek()`. This checks the new CRAN version works on this runtime, and that
its Seurat entry point is intact.


In [ ]:
library(Canek)

set.seed(1)
mk <- function(n, offset, tag) {
  m <- matrix(rpois(200 * n, lambda = 2) + offset, nrow = 200)
  rownames(m) <- paste0("gene", seq_len(200))
  colnames(m) <- paste0(tag, "_cell", seq_len(n))
  m
}

obj2 <- CreateSeuratObject(counts = cbind(mk(150, 0, "b1"), mk(150, 3, "b2")))
obj2$batch <- rep(c("b1", "b2"), each = 150)

obj2 <- NormalizeData(obj2, verbose = FALSE)
obj2 <- FindVariableFeatures(obj2, verbose = FALSE)
obj2 <- ScaleData(obj2, verbose = FALSE)
obj2 <- RunPCA(obj2, npcs = 10, verbose = FALSE)

obj2 <- RunCanek(obj2, batches = "batch", pcaDim = 10)

print(obj2)
cat("\nassays    :", paste(Assays(obj2), collapse = ", "), "\n")
cat("reductions:", paste(Reductions(obj2), collapse = ", "), "\n")
cat("Canek", as.character(packageVersion("Canek")), "ran end to end.\n")


## 8. Optional — `presto`

`presto` makes `FindAllMarkers` much faster. It is **not on CRAN**, only GitHub,
so it must be compiled here. Time it separately: if it is slow we simply drop it
and accept slower marker detection.

Skip this cell if the earlier steps already took too long.

In [ ]:
elapsed_presto <- system.time({
  install.packages("remotes")
  remotes::install_github("immunogenomics/presto", upgrade = "never", quiet = TRUE)
})[["elapsed"]]

cat("\n=== presto took", round(elapsed_presto / 60, 1), "minutes ===\n")
cat("loads:", requireNamespace("presto", quietly = TRUE), "\n")

## 9. Summary to paste back

In [ ]:
cat("R           :", R.version.string, "\n")
cat("OS          :", system("grep PRETTY_NAME /etc/os-release | cut -d'\"' -f2",
                            intern = TRUE), "\n")
cat("core install:", round(elapsed / 60, 1), "min\n")
if (exists("elapsed_bioc"))   cat("bioc + Canek:", round(elapsed_bioc / 60, 1), "min\n")
if (exists("elapsed_presto")) cat("presto      :", round(elapsed_presto / 60, 1), "min\n")
cat("Seurat      :", as.character(packageVersion("Seurat")), "\n")
cat("Canek       :", as.character(packageVersion("Canek")), "\n")
